# Isolated word decoding and sentence reconstruction

Train a CNN + residual MLP to predict a word embedding from one MEG window.
Combine several observations of each word, then use a frozen language model to reconstruct
a sentence. The neural decoder processes each window independently.

We will train SimpleB2T, evaluate it on the clinical benchmark, and try a few ablations.


Run the cells from top to bottom. With `RUN = False`, only the small examples run.
Set it to `True` to download data, train, and evaluate. Completed work is reused.

`N_SEEDS = 1` runs one training seed. Increase it to compare independent runs.
Optional experiments are enabled by name in `EXTRA`; they need the models listed in their section.


In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display
from simpleb2t.io import bundled, read, DATA
from simpleb2t.workflows import seed_list, storage_estimate

# Both notebooks share these directories. Change BASE if you keep data on another disk.
BASE = Path.cwd().resolve()
if BASE.name == "notebooks":
    BASE = BASE.parent
RAW = BASE / "raw_data"
WORK = BASE / "runs"
os.environ.setdefault("HF_HOME", str(BASE / ".cache" / "huggingface"))

N_SEEDS = 1
SEEDS = seed_list(N_SEEDS)  # 0, then 100, 200, ...
DEVICE = "cuda"
RUN = False               # True enables the main download/training/evaluation cells.
EXTRA = set()             # Optional experiments are named below.
print(f"Seeds: {SEEDS}; preview: {not RUN}; work directory: {WORK}")


## 1. Inspect the benchmark

Core and Expanded contain 100 sentences each. Every word position is assigned five distinct
held-out occurrences of that word. No occurrence is reused in another position. These are
unrelated listening occurrences: the complete clinical sentence was never spoken in a recording.

The development set uses validation recordings. Test recordings are reserved for Core and
Expanded. Exact assignments are bundled so that a new run evaluates the same benchmark.


In [ ]:
sentences = pd.read_csv(DATA / "sentences.csv")
display(sentences.head(10))
print(f"{len(sentences)} sentences; {len(bundled('vocabularies.json')['clinical'])} retrieval words")


In [ ]:
from simpleb2t.data import groups, natural
first = [g for g in groups()["test"] if g["sentence_id"] == 1]
donors = []
for position in first:
    for observation, index in enumerate(position["indices"], 1):
        item = natural()["test"]["items"][index]
        donors.append(dict(position=position["word_position"], word=position["word"],
                           observation=observation, record=item["record"], event=item["event"]))
display(pd.DataFrame(donors))


## 2. Prepare MEG and train

The notebooks share downloaded data and checkpoints. Allow 350 GiB of free disk,
64 GiB RAM, and a CUDA GPU. A 48-GB GPU can hold the FP32 Qwen model used later.

The decoder learns from individual three-second windows, using frozen T5 word embeddings
as targets and D-SigLIP as the matching loss. The best checkpoint is chosen by validation
balanced accuracy after averaging five predicted embeddings.


In [ ]:
display(storage_estimate())
from simpleb2t import data
from simpleb2t.training import run as train
if RUN:
    data.download(RAW)
    data.prepare(RAW, WORK)
    for seed in SEEDS:
        train(WORK, "ours", seed, device=DEVICE)


## 3. Understand repeated evidence

Let $z_j$ be a unit predicted embedding and $e_w$ a unit T5 target. With $k$ observations,
we compute two lexical distributions:

$$p_{\mathrm{emb}}(w)=\operatorname{softmax}_w\left(
\frac{\operatorname{normalize}(\sum_j z_j)^\top e_w}{T}\right),
\qquad p_j(w)=\operatorname{softmax}_w(z_j^\top e_w/T).$$

The brain score is
$$b(w)=\log p_{\mathrm{emb}}(w)+\frac{\alpha}{k}\sum_j\log p_j(w).$$

The first term averages embeddings. The second rewards agreement across individual observations.
The paper uses $T=0.04881238$ and $\alpha=2$. For $k=1$, the two terms are the same up to scale.
The calculation below uses random vectors only to show the shapes; it is not a decoding result.


In [ ]:
import numpy as np
import torch
from torch.nn.functional import normalize
from simpleb2t.evaluation import word_scores

generator = torch.Generator().manual_seed(0)
z = normalize(torch.randn(1, 5, 1024, generator=generator), dim=-1).numpy()
e = normalize(torch.randn(92, 1024, generator=generator), dim=-1)
score = word_scores(z, members=range(5), temperature=0.04881238, alpha=2, word_targets=e)
print("Five observation embeddings:", z.shape)
print("One score per candidate word:", score.shape)


## 4. Decode with and without the LM

Beam search adds $\lambda\log p_{\mathrm{LLM}}(s)$ to the summed brain scores.
Qwen3-8B-Base is frozen. It sees the candidate prefix generated so far, never the target prefix.
The vocabulary and number of word positions are supplied.

We use beam 50, with LM weight 1.5 at $k=1$ and 0.5 at $k=5$.
These are example experiment settings; tune them on development data for a new application.
The prompt is shown below.


In [ ]:
from simpleb2t.lm import PROMPTS
from simpleb2t.workflows import clinical, decoding_table
print(PROMPTS["E"])
if RUN:
    clinical(WORK, seeds=SEEDS, device=DEVICE, observations=(1, 5))
    table = decoding_table(WORK, SEEDS)
    main = table[table.experiment.str.startswith(("brain_ours_", "clinical_ours_")) | (table.experiment == "lm")]
    display(main[["experiment", "seed", "k", "subset", "WER", "SMR", "BAcc"]].round(2))


WER counts word substitutions, insertions, and deletions. Sentence match requires the
whole sentence to be correct. Balanced word accuracy averages recall across word classes.

At `k=1`, we evaluate each of the five observations separately and pool their errors.
At `k=5`, we combine them into one prediction. LM-only is computed once because it has
no observations to combine. Full WER pools word errors across Core and Expanded.


In [ ]:
if RUN:
    from simpleb2t.workflows import summarize
    display(summarize(main, ["model", "control", "weight", "k", "subset"], "WER").round(2))
    # Read the actual predictions, including the candidate beam.
    from simpleb2t.io import read
    examples = read(WORK / "decoding" / f"clinical_ours_{SEEDS[0]}_k5" / "predictions.json")
    display(pd.DataFrame(examples[:5])[["original_sentence_id", "reference", "prediction"]])


## 5. Optional: ablations

Add `"ablations"` to `EXTRA`. All these comparisons use Core at `k=5`:

- Add or remove the LM.
- Replace neural evidence with noise or shuffled evidence.
- Use averaged embeddings, individual evidence, or both.
- Change beam width, LM weight, or individual-evidence weight.

Noise matches each sensor/time coordinate's training mean and variance, but not correlations.
Shuffled permutes individual predicted embeddings before grouping them: the evidence stays
the same, but its association with the target words is broken.

Inspect the configurations below before running. Beams 200 and 500 take longer.


In [ ]:
from simpleb2t.workflows import ablation_plan, ablations
plan = ablation_plan()
display(pd.DataFrame(plan))
if "ablations" in EXTRA:
    ablations(WORK, seeds=SEEDS, device=DEVICE, plan=plan)
    display(decoding_table(WORK, SEEDS).round(2))


## 6. Optional: vary the number of observations

Add `"observations"` to `EXTRA`. Evaluate every subset of the five observations: 5, 10, 10,
5, and 1 subsets for $k=1,2,3,4,5$. Pool errors within a seed before comparing seeds.
The same subset of observation slots is used for every word in a sentence.


In [ ]:
if "observations" in EXTRA:
    clinical(WORK, seeds=SEEDS, device=DEVICE, observations=range(1, 6))
    curve = decoding_table(WORK, SEEDS)
    curve = curve[(curve.subset == "Core") & curve.experiment.str.startswith("clinical_ours_")]
    from simpleb2t.workflows import summarize
    display(summarize(curve, ["k"], "WER").round(2))


## 7. Optional: inspect errors and change the prompt

Choose the analyses you want by adding their names to `EXTRA`:

| Name | What it measures |
|---|---|
| `diagnostics` | Best candidate in the beam, POS accuracy, confusion counts, and sentence errors |
| `word_diagnostics` | Which words are easier to decode |
| `future_context` | How informative the later words in a source window are about the target |
| `prompts` | Prompts A–F, with and without neural evidence |

`diagnostics` reuses saved predictions. The others call Qwen and take longer.
Prompt comparisons use the supplied weights; set `published_settings=False` to fit them
on development sentences using seed 0. Reference text used in an analysis is not supplied
to the sentence decoder.


In [ ]:
from simpleb2t import analysis
if "diagnostics" in EXTRA:
    analysis.clinical_diagnostics(WORK, seeds=SEEDS)
    display(read(WORK / "analysis" / "clinical.json"))
if "word_diagnostics" in EXTRA:
    analysis.word_diagnostics(WORK, device=DEVICE, seeds=SEEDS)
if "future_context" in EXTRA:
    analysis.future_context(WORK, device=DEVICE, seeds=SEEDS)
if "prompts" in EXTRA:
    from simpleb2t.experiments import run
    display(pd.DataFrame([dict(prompt=name, text=text) for name, text in PROMPTS.items()]))
    run(WORK, "prompts", device=DEVICE, published_settings=True, seeds=SEEDS)


## 8. Optional: compare joint decoders

Add `"baselines"` to `EXTRA` to train and evaluate the original joint decoder and a version
trained on non-overlapping word windows. Each fits its temperature on validation and its LM
weight on development sentences. Both are tested at `k=1` and `k=5`.


In [ ]:
if "baselines" in EXTRA:
    from simpleb2t.experiments import run
    for model in ["joint", "stitched"]:
        for seed in SEEDS:
            train(WORK, model, seed, device=DEVICE)
    run(WORK, "clinical", device=DEVICE, published_settings=False, seeds=SEEDS)
    display(decoding_table(WORK, SEEDS).round(2))


Your predicted sentences and scores are saved under `runs/decoding/`.
See the [code guide](../docs/CODE_GUIDE.md) to change the decoder, evidence combination,
or beam search.
